In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point, LineString, Polygon
import folium
import os

plt.rcParams["font.family"] = ["SimHei", "WenQuanYi Micro Hei", "Heiti TC"]
plt.rcParams["axes.unicode_minus"] = False 

In [ ]:
airspace_zones = pd.read_csv("airspace_zones.csv")
flights = pd.read_csv("flights.csv")
trajectory_points = pd.read_csv("trajectory_points.csv")
print(airspace_zones.head())
print(airspace_zones.describe())
print(airspace_zones.info())
print(airspace_zones.dtypes)

flights = pd.read_csv("flights.csv")
print(flights.head())
print(flights.describe())
print(flights.info())
print(flights.dtypes)

trajectory_points = pd.read_csv("trajectory_points.csv")
print(trajectory_points.head())
print(trajectory_points.describe())
print(trajectory_points.info())
print(trajectory_points.dtypes)

In [ ]:
# 重复值
print(airspace_zones.duplicated().sum())
print(flights.duplicated().sum())
print(trajectory_points.duplicated().sum())

In [ ]:
# 缺失值
print(airspace_zones.isnull().sum())
print(flights.isnull().sum())
print(trajectory_points.isnull().sum())

In [ ]:
airspace_zones_gdf= gpd.GeoDataFrame(airspace_zones,geometry=gpd.GeoSeries.from_wkt(airspace_zones["zone_polygon"], crs="EPSG:4326"),crs="EPSG:4326")
trajectory_points_gdf = gpd.GeoDataFrame(trajectory_points,geometry=gpd.points_from_xy(trajectory_points["lon"],trajectory_points["lat"]),crs="EPSG:4326")
print(airspace_zones_gdf.head())
print(trajectory_points_gdf.head())

In [ ]:
# q1空域合规排查
region = gpd.sjoin(trajectory_points_gdf, airspace_zones_gdf, how="left", predicate="within")
print(f"sjoin 后原始行数：{len(region)}")  # 地基体检：会看到 5234

zone_priority = {"禁飞区": 3, "限高区": 2, "作业区": 1, "区域外": 0}
region["zone_priority"] = region["zone_type"].map(zone_priority).fillna(0)

region = region.sort_values(["flight_id", "timestamp", "zone_priority"], ascending=[True, True, False])
region = region.drop_duplicates(subset=["flight_id", "timestamp", "geometry"], keep="first")

print(f"去重后点数：{len(region)}")  # 地基体检：应该回到 5230，和原始轨迹点一致
print(f"边界重复点数：{5234 - len(region)}")

region["zone_type"] = region["zone_type"].fillna("区域外")
region["zone_id"] = region["zone_id"].fillna("NONE")
region["zone_name"] = region["zone_name"].fillna("域外空域")

region["height_tag"] = "高度合规"
region.loc[(region["zone_type"] == "限高区") & (region["altitude_m"] >= region["max_altitude_m"]), "height_tag"] = "限高超限违规"

no_fly = region[region["zone_type"] == "禁飞区"]
limit_h = region[region["zone_type"] == "限高区"]

flight_sum = region["flight_id"].nunique()
no_fly_sum = no_fly["flight_id"].nunique()
h_limit_illge = limit_h[limit_h["height_tag"] == "限高超限违规"]["flight_id"].nunique()

print(f"总航班数：{flight_sum}")
print(f"禁飞违规航班：{no_fly_sum}")
print(f"限高超限航班：{h_limit_illge}")

# 违规率
no_compliance_rate = (h_limit_illge + no_fly_sum) / flight_sum
print(f"航班违规率：{no_compliance_rate:.4f}")

compliance = flight_sum - no_fly_sum - h_limit_illge

cate = ["合规", "禁飞违规", "限高超限违规"]
counts = [compliance, no_fly_sum, h_limit_illge]  # 修正：第一根用 compliance

plt.figure(figsize=(8, 5))
bars = plt.bar(cate, counts, color=["#2ecc71", "#e74c3c", "#f39c12"])

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h + 1, f"{int(h)}", ha="center")

plt.title("航班空域合规情况统计（航班维度）")
plt.ylabel("航班数量")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# q2
# 1. 高度标签
region["height_tag"] = np.where(
    (region["zone_type"]=="限高区") & (region["altitude_m"] >= region["max_altitude_m"]),
    "限高超限违规", "高度合规"
)

# 2. 合并航班信息，转时间格式
merge_date = pd.merge(region, flights, on="flight_id", how="outer")
merge_date["timestamp"] = pd.to_datetime(merge_date["timestamp"])

# ========== 核心修复：先全量排序、差分，再判定有效性 ==========
# 按航班+时间全局排序（保留所有区域的点，不提前筛选）
merge_date = merge_date.sort_values(["flight_id", "timestamp"]).reset_index(drop=True)

# 取前一个点的区域类型
merge_date["prev_zone"] = merge_date.groupby("flight_id")["zone_type"].shift(1)
# 计算相邻两点时间差（分钟）
merge_date["dt_min"] = merge_date.groupby("flight_id")["timestamp"].diff().dt.total_seconds() / 60

# 有效作业时长判定：当前点在作业区 + 前一点也在作业区 + 高度合规
merge_date["valid_work_dt"] = np.where(
    (merge_date["zone_type"] == "作业区") & 
    (merge_date["prev_zone"] == "作业区") & 
    (merge_date["height_tag"] != "限高超限违规"),
    merge_date["dt_min"],
    0
)

# 3. 按航班聚合作业区总时长
flight_work_sum = merge_date.groupby(["flight_id", "task_type"])["valid_work_dt"].sum().reset_index()
flight_work_sum.rename(columns={"valid_work_dt":"total_work_min"}, inplace=True)

# 合并航班总时长
flight_dur = flights[["flight_id", "duration_min"]].drop_duplicates("flight_id")
flight_work_sum = flight_work_sum.merge(flight_dur, on="flight_id", how="left")

# 计算占比，除零保护
flight_work_sum["work_ratio"] = flight_work_sum["total_work_min"] / flight_work_sum["duration_min"].replace(0, np.nan)

# 4. 按任务类型聚合
task_airspace_stats = flight_work_sum.groupby("task_type").agg(
    flight_count = ("flight_id", "nunique"),
    total_work_min = ("total_work_min", "sum"),
    avg_work_min_per_flight = ("total_work_min", "mean"),
    avg_work_ratio = ("work_ratio", "mean")
).reset_index()

print(task_airspace_stats)

# 5. 柱状图
fig, ax = plt.subplots(figsize=(8,5))
bars = ax.bar(task_airspace_stats["task_type"], task_airspace_stats["avg_work_ratio"], color=["#1f77b4","#ff7f0e","#2ca02c"])

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f"{height:.3f}", ha="center", va="bottom")

ax.set_xlabel("任务类型")
ax.set_ylabel("平均作业区时长占比")
ax.set_title("不同任务类型平均作业区时长占比")
ax.set_ylim(0, 0.7)
plt.tight_layout()
plt.show()


In [ ]:
# q3 每周空域使用情况
merge_date["week"] = merge_date["timestamp"].dt.isocalendar().week
week_stats = merge_date.groupby("week").agg(
    flight_count=("flight_id", "nunique"),
    total_work_min=("valid_work_dt", "sum")
).reset_index()

print(week_stats)

fig, ax1 = plt.subplots(figsize=(9,5))

ax1.plot(week_stats["week"], week_stats["flight_count"], marker="o", color="#1f77b4", label="航班数量")
ax1.set_xlabel("周")
ax1.set_ylabel("航班数量", color="#1f77b4")
ax1.tick_params(axis="y", labelcolor="#1f77b4")

ax2 = ax1.twinx()
ax2.plot(week_stats["week"], week_stats["total_work_min"], marker="s", color="#ff7f0e", label="作业区总时长(min)")
ax2.set_ylabel("作业区总时长(min)", color="#ff7f0e")
ax2.tick_params(axis="y", labelcolor="#ff7f0e")

plt.title("各周作业区航班数量与有效作业时长")
plt.tight_layout()
plt.show()

In [ ]:
flight_work_sum["effective_ratio"] = np.where(
    flight_work_sum["task_type"] == "配送",
    1.0,
    flight_work_sum["work_ratio"]
)

# ========== 2. 各任务类型速度统计 + 分位数低速阈值 ==========
# 计算：平均速度 + 25%分位数速度（低速阈值）
type_speed_stats = merge_date.groupby("task_type")["speed_mps"].agg(
    task_avg_speed_mps = "mean",
    task_p25_speed_mps = lambda x: x.quantile(0.25)  # 下四分位数作为低速线
).reset_index()

print("各任务类型速度基准（m/s）：")
print(type_speed_stats)

# 柱状图：各任务类型平均飞行速度
fig1, ax1 = plt.subplots(figsize=(7, 5))
bars = ax1.bar(type_speed_stats["task_type"], type_speed_stats["task_avg_speed_mps"], 
               color=["#1f77b4", "#ff7f0e", "#2ca02c"])

# 柱子标注数值
for bar in bars:
    h = bar.get_height()
    ax1.text(bar.get_x()+bar.get_width()/2, h+0.05, f"{h:.2f}", ha="center")

ax1.set_xlabel("任务类型")
ax1.set_ylabel("平均飞行速度 m/s")
ax1.set_title("各任务类型平均飞行速度")
plt.tight_layout()
plt.savefig("task_avg_speed_bar.png", dpi=300, bbox_inches="tight")
plt.show()

# ========== 3. 低速航班判定（低于25%分位数） ==========
# 计算单航班平均速度
flight_speed = merge_date.groupby(["flight_id","task_type"])["speed_mps"].mean().reset_index(name="flight_avg_speed_mps")
# 合并任务类型分位数阈值
flight_speed = flight_speed.merge(type_speed_stats, on="task_type", how="left")

# 筛选：航班平均速度 < 同任务类型25%分位数速度
low_speed_flight = flight_speed[flight_speed["flight_avg_speed_mps"] < flight_speed["task_p25_speed_mps"]].reset_index(drop=True)

print("\n===== 低速航班清单（低于同任务类型25%分位数速度） =====")
print(f"共筛选出低速航班 {len(low_speed_flight)} 架次")
print(low_speed_flight)

# 水平条形图
fig2, ax2 = plt.subplots(figsize=(9,6))
ax2.barh(low_speed_flight["flight_id"], low_speed_flight["flight_avg_speed_mps"], color="#c44e52")
ax2.set_xlabel("航班平均速度 m/s")
ax2.set_ylabel("航班ID")
ax2.set_title("低速航班平均飞行速度（低于25%分位阈值）")
plt.tight_layout()
plt.savefig("low_speed_flight_barh.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# q5 空域资源运行效率
time_min = merge_date["timestamp"].min()
time_max = merge_date["timestamp"].max()
period_total_min = (time_max - time_min).total_seconds() / 60

# ========== 2. 按作业区（zone_id）分组计算利用率 ==========
# 筛选作业区相关记录，按 zone_id + zone_name 聚合
zone_airspace_stats = merge_date[merge_date["zone_type"] == "作业区"].groupby(["zone_id", "zone_name"]).agg(
    total_valid_min = ("valid_work_dt", "sum"),   # 该区累计有效占用时长
    flight_count = ("flight_id", "nunique")       # 该区内独立航班架次
).reset_index()

# 计算单区空域时间利用率
zone_airspace_stats["util_rate"] = zone_airspace_stats["total_valid_min"] / period_total_min
# 计算单区平均单架次占用时长
zone_airspace_stats["avg_per_flight_min"] = zone_airspace_stats["total_valid_min"] / zone_airspace_stats["flight_count"]

print("===== 各作业区分区空域运行指标 =====")
print(zone_airspace_stats[["zone_id", "zone_name", "total_valid_min", "flight_count", "util_rate", "avg_per_flight_min"]])

# ========== 3. 分区利用率柱状图（空管报备专用） ==========
fig, ax = plt.subplots(figsize=(8, 5))
# X轴用 zone_name 更直观，也可换成 zone_id
x_labels = zone_airspace_stats["zone_name"]
bars = ax.bar(x_labels, zone_airspace_stats["util_rate"], color="#1f77b4")

# 柱子标注利用率（小数，可按需改为百分比）
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.0005, f"{h:.4f}", ha="center")

ax.set_xlabel("作业空域")
ax.set_ylabel("空域时间利用率")
ax.set_title("各作业区空域时间利用率（报备口径）")
plt.tight_layout()
plt.savefig("zone_airspace_util_rate.png", dpi=300, bbox_inches="tight")
plt.show()

# ========== 以下保留原有整体/分任务指标作为补充 ==========
total_valid_min = merge_date["valid_work_dt"].sum()
flight_count_in_zone = merge_date[merge_date["zone_type"] == "作业区"]["flight_id"].nunique()
avg_per_flight = total_valid_min / flight_count_in_zone

print(f"\n全局统计周期总时长(min): {period_total_min:.2f}")
print(f"全部作业区累计有效占用时长(min): {total_valid_min:.2f}")
print(f"作业区内独立航班总架次: {flight_count_in_zone}")
print(f"整体平均单任务空域占用时长(min/架次): {avg_per_flight:.2f}")

# 分任务类型占用时长
type_airspace_stats = merge_date[merge_date["zone_type"] == "作业区"].groupby("task_type").agg(
    total_valid_min=("valid_work_dt", "sum"),
    flight_count=("flight_id", "nunique")
).reset_index()
type_airspace_stats["avg_per_flight_min"] = type_airspace_stats["total_valid_min"] / type_airspace_stats["flight_count"]

print("\n===== 分任务类型空域占用指标 =====")
print(type_airspace_stats)

In [ ]:
region["track_status"] = "合规"
region.loc[region["zone_type"] == "禁飞区", "track_status"] = "禁飞违规"
region.loc[(region["zone_type"] == "限高区") & (region["height_tag"] == "限高超限违规"), "track_status"] = "限高超限违规"

region = region.sort_values(["flight_id", "timestamp"]).to_crs(epsg=4326)

def build_line(group):
    points = group["geometry"].tolist()
    return LineString(points) if len(points) >= 2 else None

track_lines = region.groupby(["flight_id", "track_status"]).apply(build_line).dropna()
track_gdf = gpd.GeoDataFrame(
    track_lines.reset_index(),
    geometry=0,
    crs=4326
).rename(columns={0:"geometry"})

# ========== 2. 空域边界数据预处理 ==========
airspace_zones = airspace_zones_gdf.to_crs(epsg=4326)

# 空域类型 → 样式映射
zone_color_map = {
    "禁飞区": "#e74c3c",   # 红色：禁飞区，与违规轨迹同色，越界一目了然
    "限高区": "#f39c12",   # 橙色：限高区
    "作业区": "#3498db",   # 蓝色：作业区
    "区域外": "#95a5a6"
}

# ========== 3. 计算地图中心点 ==========
lon_mean = region["geometry"].apply(lambda p: p.x).mean()
lat_mean = region["geometry"].apply(lambda p: p.y).mean()
center = [lat_mean, lon_mean]

# 初始化地图
m = folium.Map(location=center, zoom_start=12, tiles="CartoDB positron")

# ========== 4. 底层：folium.GeoJson 叠加空域边界面 ==========
def zone_style(feature):
    zone_type = feature["properties"].get("zone_type", "区域外")
    color = zone_color_map.get(zone_type, "#95a5a6")
    return {
        "color": color,          # 边框颜色
        "weight": 1.5,           # 边框粗细
        "fillColor": color,      # 填充颜色
        "fillOpacity": 0.2,      # 填充透明度
        "dashArray": None
    }

zone_layer = folium.FeatureGroup(name="空域边界", show=True)
folium.GeoJson(
    airspace_zones,
    style_function=zone_style,
    popup=folium.GeoJsonPopup(
        fields=["zone_id", "zone_name", "zone_type", "max_altitude_m"],
        aliases=["空域编号", "空域名称", "类型", "高度上限(m)"],
        max_width=250
    )
).add_to(zone_layer)
zone_layer.add_to(m)

# ========== 5. 上层：三色飞行轨迹线 ==========
track_layer = folium.FeatureGroup(name="飞行轨迹", show=True)
track_color_map = {"合规": "#2ecc71", "禁飞违规": "#e74c3c", "限高超限违规": "#f39c12"}

for _, row in track_gdf.iterrows():
    coords = [(p[1], p[0]) for p in row.geometry.coords]
    folium.PolyLine(
        locations=coords,
        color=track_color_map[row["track_status"]],
        weight=2.5,
        opacity=0.9,
        popup=folium.Popup(f"航班：{row['flight_id']}<br>状态：{row['track_status']}", max_width=200)
    ).add_to(track_layer)
track_layer.add_to(m)

# ========== 6. 图层控制 ==========
folium.LayerControl(position="topright").add_to(m)

# 保存并输出
m.save("flight_track_with_zones.html")
print("带空域边界的轨迹地图已生成：flight_track_with_zones.html")
m